In [1]:
# Cell 1:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output, dash_table  # dash_table is included here

# Load CSV
df_csv = pd.read_csv("aac_shelter_outcomes (1).csv")

# Filter mapping 
filter_map = {
    "Reset": df_csv,
    "Water Rescue": df_csv[df_csv["animal_type"] == "Dog"],  # Example condition
    "Mountain or Wilderness Rescue": df_csv[df_csv["animal_type"] == "Dog"],  # Adjust as needed
    "Disaster Rescue or Individual Tracking": df_csv[df_csv["animal_type"] == "Dog"],  # Adjust as needed
}


In [2]:
# Cell 2: 
app = Dash(__name__)


In [3]:
# Cell 3: Dashboard 
app.layout = html.Div([
    html.H1("AAC Shelter Outcomes Dashboard"),
    
    # Branding
    html.Div([
        html.A(html.Img(src="https://upload.wikimedia.org/wikipedia/commons/1/1f/SNHU_Logo.png", height="50px"),
               href="https://www.snhu.edu"),
        html.Span("Dashboard by Ethan Chapman", style={"margin-left": "20px", "font-weight": "bold"})
    ], style={"display": "flex", "align-items": "center"}),
    
    # Filters
    html.Div([
        html.Label("Select Rescue Filter:"),
        dcc.Dropdown(
            id='filter-dropdown',
            options=[{"label": k, "value": k} for k in filter_map.keys()],
            value="Reset",
            clearable=False
        ),
        html.Br(),
        html.Label("Filter by Age (weeks):"),
        dcc.RangeSlider(
            id='age-slider',
            min=0,
            max=df_csv['age_upon_outcome_in_weeks'].max(),
            step=1,
            value=[0, df_csv['age_upon_outcome_in_weeks'].max()],
            marks={0: '0', int(df_csv['age_upon_outcome_in_weeks'].max()): str(int(df_csv['age_upon_outcome_in_weeks'].max()))},
            tooltip={"placement": "bottom", "always_visible": True}
        )
    ], style={"width": "50%", "margin-bottom": "20px"}),
    
    # Data table
    dash_table.DataTable(
        id='data-table',
        columns=[{"name": i, "id": i} for i in df_csv.columns],
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'}
    ),
    
    html.Br(),
    
    # Charts
    html.Div([
        dcc.Graph(id='pie-chart'),
        dcc.Graph(id='geo-map')
    ])
])


In [4]:
# Cell 4: Callbacks 
@app.callback(
    Output('data-table', 'data'),
    Output('pie-chart', 'figure'),
    Output('geo-map', 'figure'),
    Input('filter-dropdown', 'value'),
    Input('age-slider', 'value')
)
def update_dashboard(selected_filter, age_range):
    # Filter the data by selected rescue type
    filtered_df = filter_map[selected_filter]
    
    # Further filter by age range
    filtered_df = filtered_df[
        (filtered_df['age_upon_outcome_in_weeks'] >= age_range[0]) &
        (filtered_df['age_upon_outcome_in_weeks'] <= age_range[1])
    ]
    
    # Update data table
    table_data = filtered_df.to_dict('records')
    
    # Update pie chart
    pie_fig = px.pie(
        filtered_df,
        names='animal_type',
        title='Animal Type Distribution'
    )
    
    # Update geolocation map
    geo_fig = px.scatter_mapbox(
        filtered_df,
        lat='location_lat',
        lon='location_long',
        color='animal_type',
        hover_name='name',
        hover_data=['breed', 'age_upon_outcome_in_weeks', 'outcome_type'],
        zoom=10,
        height=500
    )
    geo_fig.update_layout(mapbox_style="open-street-map")
    
    return table_data, pie_fig, geo_fig


In [5]:
# Cell 5: 
if __name__ == "__main__":
    app.run(port=8050, debug=True)  
